# Dataset statistics

Count and group-by queries with matplotlib charts. Use this
notebook to get a feel for how the corpus is distributed across
agent types, ego actions, and environments before you go
looking for specific scenarios.

> See `01_quickstart.ipynb` for an introduction to the dataset
> and `02_query_dsl_tour.ipynb` for the query language itself.


## Setup


In [ ]:
import os
from pathlib import Path

from causal_ai_av.dataset import CausalAVDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [ ]:
ds = CausalAVDataset(Path(os.environ["CAUSAL_AV_DATASET_ROOT"]))
print(f"corpus loaded — {len(ds)} clips")


## Headline counts

Single-query counts for common driving primitives. `count()`
returns the number of *clips* with at least one match, not the
number of matching entities.


In [ ]:
rows = [
    ("pedestrian",              ds.count("agent.type = ped")),
    ("vehicle",                 ds.count("agent.type = vehicle")),
    ("VRU (ped/cyclist)",       ds.count("agent.type = vru")),
    ("cyclist",                 ds.count("agent.type = cyclist")),
    ("stop sign",               ds.count("obj.type = stop_sign")),
    ("traffic light (red)",     ds.count("light.color = red")),
    ("crosswalk environment",   ds.count("env.type = crosswalk")),
    ("intersection environment", ds.count("env.type = intersection")),
    ("roundabout environment",  ds.count("env.type = roundabout")),
    ("multi-lane road",         ds.count("env(type = road, lanes >= 2)")),
    ("ego decelerates",         ds.count("ego.action = decel")),
    ("ego stops at red",        ds.count("light.color = red and ego.action = stop")),
]
headline = pd.DataFrame(rows, columns=["query", "clips"]).sort_values(
    "clips", ascending=False, ignore_index=True
)
headline


## Distribution of agent types

`group_by(query, key)` runs the query and buckets matches by
the value of `key`. Values returned are clip counts (one per
distinct clip with a matching entity).


In [ ]:
agent_dist = ds.group_by(
    "agent.type = vehicle or agent.type = vru or agent.type = animal",
    key="agent.type",
)
s = pd.Series(agent_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#3b6ea5")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per agent type")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()


## Distribution of ego actions


In [ ]:
ego_dist = ds.group_by(
    "ego.action.type = drive or ego.action.type = stop or ego.action.type = decel "
    "or ego.action.type = yield or ego.action.type = turn_left "
    "or ego.action.type = turn_right or ego.action.type = change_lane",
    key="ego.action.type",
)
s = pd.Series(ego_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#4f7d4a")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per ego-action type")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()


## Distribution of environment types


In [ ]:
env_dist = ds.group_by(
    "env.type = road or env.type = intersection or env.type = crosswalk "
    "or env.type = sidewalk or env.type = roundabout or env.type = cycle_lane "
    "or env.type = tunnel",
    key="env.type",
)
s = pd.Series(env_dist, name="clips").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 0.45 * max(len(s), 4) + 1.5))
s.plot(kind="barh", ax=ax, color="#8a5a3b")
ax.set_xlabel("clips")
ax.set_ylabel("")
ax.set_title("Clips per environment type")
for i, v in enumerate(s.values):
    ax.text(v + max(s.values) * 0.01, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.show()


## Composite questions

DSL queries compose, so you can pose questions that combine
primitives — for example, "of the clips that contain a red
light, what fraction also contain an ego stop?"


In [ ]:
n_red          = ds.count("light.color = red")
n_red_stop     = ds.count("light.color = red and ego.action = stop")
n_yellow_clear = ds.count("light(color = yellow, could_have_cleared = true)")
n_jaywalk      = ds.count("agent(type = ped, action(jaywalk = true))")

composite = pd.DataFrame([
    ("ego stops at red light", n_red_stop, n_red, n_red_stop / max(n_red, 1)),
    ("yellow ego could have safely cleared", n_yellow_clear, None, None),
    ("jaywalking pedestrian present", n_jaywalk, None, None),
], columns=["question", "matching clips", "denominator", "ratio"])
composite.style.format({"ratio": "{:.0%}"}, na_rep="—")
